# AI/ML Security Foundations - Hands-On Lab

**Welcome!** This 45–60 minute hands-on lab will introduce you to fundamental machine learning concepts from a security perspective.

**What You'll Learn:**
1. How to work with a simple dataset (the building block of all AI/ML)
2. How to train a basic machine learning model
3. How to interpret predictions and evaluation metrics
4. What happens when training data is tampered with (label poisoning attack)
5. How to verify your Python environment for the rest of the course

**By the end of this lab, you will:**
- Load and explore a dataset (Iris flower dataset)
- Visualize feature distributions to understand the data
- Train a Logistic Regression classifier (a simple, interpretable model)
- Evaluate model accuracy using standard metrics
- Simulate a data poisoning attack and measure its impact on model performance

**Security Context:**  
Understanding how supervised models learn patterns from labeled training data builds intuition for later modules on data poisoning, adversarial attacks, and model robustness. When attackers compromise training data, model behavior changes—sometimes in subtle ways that are hard to detect.

**No Data Science Background Required!** All code is explained step-by-step. If you get stuck, read the "Troubleshooting" sections.

# Step 1: Environment Verification

**What we're doing:** Checking that your Python environment has all required libraries installed.

**Why this matters:** Machine learning requires specific libraries for data processing (pandas, numpy), visualization (matplotlib, seaborn), and model training (scikit-learn). If any are missing, the lab won't run.

**What to expect:** The next cell will check each package and report "OK" or "MISSING". If everything is OK, proceed to Step 2. If packages are missing, see the troubleshooting section below.

In [ ]:
# Environment verification - checks if required libraries are installed
import importlib, sys, platform

# List of required Python packages for this lab
required = ["numpy", "pandas", "matplotlib", "seaborn", "sklearn"]
status = {}

# Try to import each package and record status
for pkg in required:
    try:
        importlib.import_module(pkg)
        status[pkg] = "✓ OK"
    except Exception as e:
        status[pkg] = f"✗ MISSING ({e.__class__.__name__})"

# Display environment information
print("=" * 50)
print("ENVIRONMENT CHECK")
print("=" * 50)
print(f"Python version: {sys.version.split()[0]}")
print(f"Platform:       {platform.platform()}")
print()
print("Package status:")
for k, v in status.items():
    print(f"  {k:12}: {v}")
print("=" * 50)

# Check if any packages are missing
missing = [k for k, v in status.items() if "MISSING" in v]
if missing:
    print("\n⚠ ATTENTION: Some packages are missing!")
    print("   Please see the troubleshooting section below.")
else:
    print("\n✓ All packages installed. Environment is ready!")
    print("  Proceed to Step 2.")

# Troubleshooting (if packages are missing)
If the previous cell reported missing packages, open a terminal and run:

```pwsh
python -m pip install numpy pandas matplotlib seaborn scikit-learn
```
or run this cell in the notebook:


In [ ]:

%pip install numpy pandas matplotlib seaborn scikit-learn

# Step 2: Load & Inspect the Dataset

**What we're doing:** Loading a simple, well-known dataset to practice with.

**Dataset:** The Iris flower dataset contains measurements from 150 flowers across 3 species. It's a classic "hello world" dataset for machine learning—small, clean, and easy to understand.

**Dataset Details:**
- **150 samples** (flower measurements)
- **4 numeric features** (measurements in centimeters):
  - Sepal length (outer petal)
  - Sepal width
  - Petal length (inner petal)
  - Petal width
- **3 classes** (species):
  - Setosa
  - Versicolor  
  - Virginica

**Task:** Predict which species a flower belongs to based on its measurements. This is called **multiclass classification**.

**Security note:** In real-world scenarios, attackers might tamper with training data to manipulate model predictions. Understanding how models learn from data helps you spot when something's wrong.

In [ ]:
# Load the Iris dataset and examine its structure
from sklearn import datasets
import pandas as pd
import numpy as np

# Load dataset (built into scikit-learn, no download needed)
iris = datasets.load_iris()

# X = features (measurements), y = labels (species)
# We use pandas DataFrames for easier viewing
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target, name="species")
class_names = iris.target_names

print("Dataset loaded successfully!")
print("=" * 50)
print(f"Dataset shape: {X.shape[0]} samples × {X.shape[1]} features")
print(f"Classes (species): {', '.join(class_names)}")
print("=" * 50)
print("\nFirst 5 samples:")
display(X.head())
print("\nTarget distribution (how many of each species):")
print(y.value_counts().sort_index())
print("\nNote: 0=setosa, 1=versicolor, 2=virginica")

In [ ]:
# Visualize the data to understand patterns
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

print("Statistical summary of features:")
print("=" * 50)
display(X.describe())
print("\nWhat this tells us:")
print("- Mean: average value for each measurement")
print("- Std: how spread out the values are")
print("- Min/Max: range of values")
print("=" * 50)

# Show distribution of each feature
print("\nGenerating histograms...")
X.hist(figsize=(10, 6), bins=15, edgecolor='black')
plt.suptitle("Feature Distributions (How measurements vary)", fontsize=14)
plt.tight_layout()
plt.show()

# Pairplot shows relationships between features
print("\nGenerating pairplot (this shows how features relate to each other)...")
print("Tip: Look for clear separations between colored groups—this means the model can distinguish species easily.")
pair_df = X.copy()
pair_df['species'] = y.map({i: name for i, name in enumerate(class_names)})
sns.pairplot(pair_df, hue='species', diag_kind='hist', height=2)
plt.show()
print("\n✓ Visualization complete. Notice how some feature pairs separate species better than others.")

# Step 3: Train/Test Split

**What we're doing:** Dividing our dataset into two parts:
- **Training set (75%):** Used to teach the model
- **Test set (25%):** Used to evaluate how well the model generalizes to new data

**Why this matters:**  
If we train and test on the same data, we can't tell if the model truly learned patterns or just memorized the training examples. The test set simulates "unseen" data.

**Security relevance:**  
Attackers often manipulate training data. By keeping a clean test set, we can detect when model performance degrades unexpectedly—a potential sign of poisoning.

In [ ]:
# Split data into training (75%) and test (25%) sets
from sklearn.model_selection import train_test_split

# stratify=y ensures each set has the same proportion of each species
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.25,      # 25% for testing
    random_state=42,     # fixed seed for reproducibility
    stratify=y           # maintain class balance
)

print("Data split complete!")
print("=" * 50)
print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")
print("=" * 50)
print("\nTraining set class distribution:")
print(y_train.value_counts().sort_index())
print("\nTest set class distribution:")
print(y_test.value_counts().sort_index())
print("\n✓ Both sets have balanced classes (good for training!)")

# Step 4: Train a Logistic Regression Model

**What we're doing:** Training a machine learning model to predict flower species from measurements.

**Model choice:** Logistic Regression is a simple, interpretable linear model. It learns a "weight" for each feature that indicates how much that measurement influences the prediction.

**How it works (simplified):**
- Model looks at training examples and their correct labels
- Adjusts weights to minimize prediction errors
- Higher absolute weight = that feature is more important for classification

**Why Logistic Regression for security:**
- **Transparent:** We can inspect weights to understand model decisions
- **Fast:** Trains quickly, making it practical for testing
- **Baseline:** Good starting point before trying more complex models

**Security note:** More complex models (deep neural networks) are harder to interpret, making it easier for attackers to hide malicious behavior.

In [ ]:
# Train a Logistic Regression classifier
from sklearn.linear_model import LogisticRegression

print("Training model...")
# Create and train the model
logreg = LogisticRegression(
    max_iter=500,              # maximum training iterations
    multi_class='multinomial'  # handles 3+ classes properly
)
logreg.fit(X_train, y_train)
print("✓ Training complete!\n")

# Examine learned coefficients (weights)
print("=" * 50)
print("MODEL COEFFICIENTS (Feature Importance)")
print("=" * 50)
coef_df = pd.DataFrame(logreg.coef_, columns=X.columns)
coef_df['class'] = [f"{name}" for name in class_names]
print("\nInterpretation:")
print("- Positive weight: higher feature value → more likely this class")
print("- Negative weight: higher feature value → less likely this class")
print("- Larger absolute value: stronger influence on prediction\n")
display(coef_df.set_index('class'))

print("\nIntercepts (baseline predictions before considering features):")
print(logreg.intercept_)

In [ ]:
# Visualize feature weights to see which measurements matter most
plt.figure(figsize=(10, 5))
coef_long = coef_df.melt(id_vars='class', var_name='feature', value_name='weight')
sns.barplot(data=coef_long, x='feature', y='weight', hue='class')
plt.title('Feature Importance by Species (Logistic Regression Coefficients)', fontsize=12)
plt.xlabel('Feature (Measurement)')
plt.ylabel('Weight (Importance)')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Species')
plt.tight_layout()
plt.show()

print("\nKey observation: Petal measurements (length/width) have stronger weights")
print("than sepal measurements, meaning they're more useful for classification.")

# Step 5: Evaluate Model Performance

**What we're doing:** Testing how well our trained model predicts species on the test set (data it has never seen).

**Metrics we'll examine:**
- **Accuracy:** Percentage of correct predictions (e.g., 95% = 95 out of 100 correct)
- **Precision:** Of all predicted species X, how many were truly X? (avoids false alarms)
- **Recall:** Of all actual species X, how many did we find? (avoids missing real cases)
- **Confusion Matrix:** Table showing where the model was right/wrong

**Security context:**  
Monitoring these metrics over time helps detect model degradation—a key indicator of data poisoning or adversarial attacks. Sudden drops in accuracy can signal that training data was compromised.

In [ ]:
# Make predictions on the test set
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

print("Making predictions on test set...")
y_pred = logreg.predict(X_test)           # predicted class labels
y_proba = logreg.predict_proba(X_test)    # prediction probabilities

print("✓ Predictions complete!\n")
print("=" * 50)
print("SAMPLE PREDICTIONS (First 5 test samples)")
print("=" * 50)
print("Prediction probabilities (columns = setosa, versicolor, virginica):")
display(pd.DataFrame(y_proba[:5], columns=class_names).round(3))
print("\nInterpretation: Each row sums to 1.0 (100% probability distributed across classes)")
print("The class with highest probability is the final prediction.\n")

# Performance metrics
print("=" * 50)
print("CLASSIFICATION REPORT")
print("=" * 50)
print(classification_report(y_test, y_pred, target_names=class_names))
print("\nHow to read this:")
print("- Precision: % of predictions for this class that were correct")
print("- Recall: % of actual instances of this class that were found")
print("- F1-score: Harmonic mean of precision and recall (balanced metric)")
print("- Support: Number of actual instances of each class in test set\n")

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(cmap='Blues', ax=ax)
plt.title('Confusion Matrix (Where did the model make mistakes?)', fontsize=12)
plt.tight_layout()
plt.show()
print("\nHow to read confusion matrix:")
print("- Diagonal (top-left to bottom-right): CORRECT predictions")
print("- Off-diagonal: ERRORS (predicted X but was actually Y)")

# Step 6: Visualize Decision Boundaries (Optional)

**What we're doing:** Creating a 2D visualization to see how the model separates species.

**Note:** Our full model uses all 4 features, but we can only visualize 2 dimensions at once. This is a simplified view for learning purposes—we'll retrain using just petal length and petal width.

**What you'll see:**  
Colored regions showing which species the model would predict for any combination of petal measurements. Dots are actual flowers from the dataset.

**Security insight:**  
Understanding decision boundaries helps you see how slight changes in input features might push a prediction across a boundary—this is the basis of adversarial attacks we'll cover in Chapter 3.

In [ ]:
# Visualize decision boundary using only 2 features
from sklearn.linear_model import LogisticRegression

print("Training a 2D model for visualization...")

# Use only petal measurements (the two most important features)
feat_pair = ['petal length (cm)', 'petal width (cm)']
X2 = X[feat_pair]

# Split and train using just these 2 features
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y, test_size=0.25, random_state=42, stratify=y
)
model2 = LogisticRegression(max_iter=500, multi_class='multinomial')
model2.fit(X2_train, y2_train)
print("✓ 2D model trained\n")

# Create a grid of points covering the feature space
x_min, x_max = X2[feat_pair[0]].min() - 0.5, X2[feat_pair[0]].max() + 0.5
y_min, y_max = X2[feat_pair[1]].min() - 0.5, X2[feat_pair[1]].max() + 0.5
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 300),
    np.linspace(y_min, y_max, 300)
)

# Predict class for every point in the grid
print("Generating decision regions (this creates the colored background)...")
Z = model2.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

# Plot decision regions and actual data points
plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.3, cmap='Pastel1', levels=2)

scatter_df = X2.copy()
scatter_df['species'] = y.map({i: name for i, name in enumerate(class_names)})
sns.scatterplot(
    data=scatter_df, 
    x=feat_pair[0], 
    y=feat_pair[1], 
    hue='species', 
    edgecolor='black',
    s=60,
    alpha=0.8
)
plt.title('Decision Boundaries (Petal Length vs Petal Width)', fontsize=13)
plt.xlabel(feat_pair[0])
plt.ylabel(feat_pair[1])
plt.legend(title='Species', loc='upper left')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("- Colored regions: model's prediction for that area of the graph")
print("- Dots: actual flowers (colored by true species)")
print("- Notice how the model draws lines to separate species")
print("- In Chapter 3, you'll learn how attackers craft inputs near these boundaries!")

# Step 7: Simulate a Label Poisoning Attack

**What we're doing:** Deliberately corrupting training labels to see how it affects model performance.

**Attack scenario:**  
An attacker gains access to your training dataset and flips a small percentage of labels to the wrong species. This is called **label poisoning** or **label flipping**.

**How the attack works:**
1. We randomly select 7% of training samples
2. Change their labels to incorrect species (0→1, 1→2, 2→0)
3. Retrain the model on poisoned data
4. Compare accuracy on clean test set

**Real-world impact:**  
Even small amounts of poisoning can degrade model performance. In security-critical applications (fraud detection, malware classification), attackers use this to evade detection.

**Security lesson:**  
Data integrity is fundamental to AI security. Always validate training data sources and monitor for unexpected label patterns.

In [ ]:
# Simulate a label poisoning attack
print("SIMULATING LABEL POISONING ATTACK")
print("=" * 50)

# Attack parameters
poison_fraction = 0.07  # 7% of training data will be corrupted
poison_count = int(len(y_train) * poison_fraction)

# Randomly select samples to poison
np.random.seed(123)
indices = np.random.choice(y_train.index, size=poison_count, replace=False)

# Create poisoned labels (flip to wrong class)
y_train_poisoned = y_train.copy()
label_map = {0: 1, 1: 2, 2: 0}  # setosa→versicolor, versicolor→virginica, virginica→setosa
y_train_poisoned.loc[indices] = y_train_poisoned.loc[indices].map(label_map)

print(f"Poisoned {poison_count} out of {len(y_train)} labels ({poison_fraction*100:.1f}%)")
print("\nOriginal training label counts:")
print(y_train.value_counts().sort_index())
print("\nPoisoned training label counts:")
print(y_train_poisoned.value_counts().sort_index())
print("\n⚠ Notice: Label distribution has shifted slightly due to flips\n")

# Train a new model on poisoned data
print("Training model on POISONED data...")
logreg_poisoned = LogisticRegression(max_iter=500, multi_class='multinomial')
logreg_poisoned.fit(X_train, y_train_poisoned)
print("✓ Poisoned model trained\n")

# Evaluate both models on CLEAN test set
y_pred_clean = logreg.predict(X_test)
y_pred_poisoned = logreg_poisoned.predict(X_test)

from sklearn.metrics import accuracy_score
clean_acc = accuracy_score(y_test, y_pred_clean)
poisoned_acc = accuracy_score(y_test, y_pred_poisoned)

print("=" * 50)
print("ATTACK IMPACT")
print("=" * 50)
print(f"Clean model test accuracy:    {clean_acc:.4f} ({clean_acc*100:.1f}%)")
print(f"Poisoned model test accuracy: {poisoned_acc:.4f} ({poisoned_acc*100:.1f}%)")
print(f"Accuracy drop:                {(clean_acc - poisoned_acc):.4f} ({(clean_acc - poisoned_acc)*100:.1f} percentage points)")
print("=" * 50)

if poisoned_acc < clean_acc:
    print("\n⚠ POISONING SUCCESSFUL: Model performance degraded!")
    print("Even 7% label corruption noticeably reduced accuracy.")
else:
    print("\n✓ Model remained robust (but this won't always be true)")

# Detailed comparison
print("\nDetailed classification report (poisoned model):")
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred_poisoned, target_names=class_names))

# Step 8: Model Complexity vs. Performance (Optional)

**What we're doing:** Comparing Logistic Regression to Decision Trees to understand model complexity.

**Decision Trees:**  
Another type of classifier that makes decisions like a flowchart. They can be very simple (few splits) or very complex (many splits).

**Why this matters for security:**
- **Simple models (shallow trees):** Easy to understand and audit, but may miss subtle patterns
- **Complex models (deep trees):** More powerful but can overfit (memorize training data instead of learning patterns)
- **Overfitting risk:** Makes models vulnerable to adversarial manipulation

**What you'll see:**  
We'll train trees of different depths and plot accuracy. Watch for the "sweet spot" where accuracy peaks before overfitting kicks in.

In [ ]:
# Train Decision Trees with varying complexity (depth)
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

print("Training Decision Trees with different max depths...")
results = []

for depth in range(1, 11):
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_train, y_train)
    test_acc = accuracy_score(y_test, dt.predict(X_test))
    train_acc = accuracy_score(y_train, dt.predict(X_train))
    results.append((depth, train_acc, test_acc))

res_df = pd.DataFrame(results, columns=['max_depth', 'train_accuracy', 'test_accuracy'])
print("✓ Training complete\n")

print("=" * 50)
print("DEPTH vs ACCURACY")
print("=" * 50)
display(res_df)
print("\nKey observations:")
print("- Training accuracy: How well model fits training data")
print("- Test accuracy: How well model generalizes to new data")
print("- When train >> test: OVERFITTING (model memorized training data)")

# Visualize the trade-off
plt.figure(figsize=(8, 5))
plt.plot(res_df['max_depth'], res_df['train_accuracy'], marker='o', label='Training Accuracy', linewidth=2)
plt.plot(res_df['max_depth'], res_df['test_accuracy'], marker='s', label='Test Accuracy', linewidth=2)
plt.xlabel('Tree Depth (Model Complexity)', fontsize=11)
plt.ylabel('Accuracy', fontsize=11)
plt.title('Model Complexity vs. Performance', fontsize=13)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\nSecurity takeaway:")
print("- More complex ≠ better. Deep models can memorize noise and become brittle.")
print("- Simpler models are often more robust to small data perturbations.")
print("- In later chapters, you'll see how attackers exploit overfitted models.")

# Step 9: Security Reflection & Next Steps

**Congratulations!** You've completed the foundational lab. Let's review what you learned:

## Key Security Lessons

1. **Data is Everything**  
   - ML models are only as good as their training data
   - Poisoning just 7% of labels degraded our model's accuracy
   - Real attacks can be more subtle and harder to detect

2. **Model Transparency Matters**  
   - Logistic Regression showed us exactly which features influenced decisions
   - Complex models (deep neural networks) hide these details
   - Interpretability helps security teams audit AI systems

3. **Complexity Has Costs**  
   - Deeper models can memorize noise (overfitting)
   - Overfitted models are more vulnerable to adversarial manipulation
   - Balance accuracy with robustness

4. **Testing Catches Problems**  
   - Keeping a clean test set helped us detect poisoning impact
   - Monitoring metrics over time reveals when models degrade
   - Sudden performance drops signal potential attacks

## What's Next in the Course

- **Chapter 2:** Data poisoning attacks in depth (backdoors, targeted attacks)
- **Chapter 3:** Adversarial examples (crafting inputs that fool models)
- **Chapter 4:** Privacy attacks (extracting training data from models)
- **Chapter 5:** Secure AI integration (defending production systems)
- **Chapter 6:** Enterprise AI security strategy

## Try on Your Own (Optional Challenges)

1. **Increase poison fraction to 20%** — How much does accuracy drop?
2. **Poison only one class** (e.g., flip only setosa labels) — Does the attack target specific predictions?
3. **Add random noise to features** — How robust is the model to measurement errors?
4. **Compare clean vs poisoned model coefficients** — Can you detect which features changed most?

## Environment Check

If all cells ran successfully, your environment is ready for the rest of the course.  
If you encountered errors, revisit the troubleshooting section in Step 1 or ask your instructor.

**You're ready to dive deeper into AI security threats and defenses!**

In [ ]:
# BONUS: Deep dive into clean vs poisoned model differences
import pandas as pd
from sklearn.metrics import confusion_matrix

print("BONUS ANALYSIS: Comparing Clean vs Poisoned Models")
print("=" * 50)

# Compare coefficients
clean_coef = pd.DataFrame(logreg.coef_, columns=X.columns)
clean_coef['model'] = [f"clean_{n}" for n in class_names]

poison_coef = pd.DataFrame(logreg_poisoned.coef_, columns=X.columns)
poison_coef['model'] = [f"poison_{n}" for n in class_names]

comparison = pd.concat([clean_coef, poison_coef])
print("\nModel Coefficients Comparison:")
display(comparison.set_index('model'))

# Calculate coefficient differences
coef_diff = poison_coef.drop(columns='model') - clean_coef.drop(columns='model')
coef_diff['class'] = class_names
print("\nCoefficient Changes (poisoned - clean):")
print("Larger values = bigger impact of poisoning on that feature")
display(coef_diff.set_index('class'))

# Compare confusion matrices
cm_clean = confusion_matrix(y_test, y_pred_clean)
cm_poison = confusion_matrix(y_test, y_pred_poisoned)

print("\nClean Model Confusion Matrix:")
print(pd.DataFrame(cm_clean, index=class_names, columns=class_names))

print("\nPoisoned Model Confusion Matrix:")
print(pd.DataFrame(cm_poison, index=class_names, columns=class_names))

print("\nDifference (poisoned - clean):")
print("Positive numbers = more misclassifications")
diff_cm = cm_poison - cm_clean
print(pd.DataFrame(diff_cm, index=class_names, columns=class_names))

print("\n" + "=" * 50)
print("INTERPRETATION GUIDE")
print("=" * 50)
print("If you see large coefficient changes:")
print("→ The poisoning forced the model to rely differently on features")
print("\nIf confusion matrix differences are mostly off-diagonal:")
print("→ The poisoned model makes more classification errors")
print("\nThis deep dive shows HOW poisoning changes the model internally,")
print("not just the final accuracy number.")